# HyDE and Parent Document Retrieval

In the previous notebooks, we built increasingly sophisticated retrieval pipelines. Now we'll tackle two fundamental problems: the mismatch between how users phrase queries and how documents are written (HyDE), and the trade-off between precise matching and sufficient context (Parent Document Retrieval).

Standard retrieval has a fundamental mismatch: **queries are questions, documents are answers**. When you embed "What is photosynthesis?" and "Photosynthesis is the process by which plants convert sunlight...", they end up in different regions of vector space despite being semantically related.

This notebook covers two techniques that address this problem:

1. **HyDE (Hypothetical Document Embeddings)**: Generate a hypothetical answer first, then embed that instead of the query
2. **Parent Document Retrieval**: Index small chunks for precision, but return their parent documents for context

Both techniques add overhead but significantly improve retrieval quality in different scenarios.

In [ ]:
# DEPENDENCY: pip install -q openai numpy
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
import numpy as np
from getpass import getpass
from dataclasses import dataclass, field
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

## Part 1: HyDE (Hypothetical Document Embeddings)

### The Problem

Embedding models are trained on semantic similarity, but questions and answers are semantically *related*, not *similar*:

- Query: "How does TCP handle packet loss?"
- Document: "TCP uses acknowledgments and retransmission timers. When a sender doesn't receive an ACK within the timeout period, it retransmits the packet..."

These express the same concept but use different vocabulary and structure.

### The Solution: HyDE

1. Take the user's query
2. Ask an LLM to generate a *hypothetical* answer (doesn't need to be accurate)
3. Embed the hypothetical answer
4. Use that embedding to search

The hypothetical answer is in "document space", so it matches real documents better.

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def generate_hypothetical_answer(query: str) -> str:
    """Generate a hypothetical document that would answer the query."""
    response = client.responses.create(
        model=MODEL,
        instructions="""Generate a hypothetical document passage that would answer the given question.
The passage should be written as if it's from an authoritative source like a textbook or documentation.
It doesn't need to be perfectly accurate - the goal is to match the style and vocabulary of real documents.
Keep it to 2-3 sentences.""",
        input=query
    )
    return response.output_text

# Demonstrate HyDE
query = "How does TCP handle packet loss?"
hypothetical = generate_hypothetical_answer(query)

print(f"Query: {query}")
print(f"\nHypothetical Answer: {hypothetical}")

In [ ]:
# Sample documents (simulating a knowledge base)
documents = [
    "TCP uses a sliding window protocol with acknowledgments. When packets are lost, "
    "the sender detects this through timeout or duplicate ACKs and retransmits the missing segments.",
    
    "UDP is a connectionless protocol that does not guarantee delivery. "
    "It's faster than TCP but provides no built-in mechanism for handling lost packets.",
    
    "The HTTP protocol operates at the application layer and typically runs over TCP. "
    "HTTP/2 introduced multiplexing to improve performance over single connections.",
    
    "Network congestion occurs when too many packets are sent into the network. "
    "TCP's congestion control algorithms like CUBIC adjust the sending rate dynamically.",
    
    "The three-way handshake (SYN, SYN-ACK, ACK) establishes a TCP connection. "
    "This ensures both parties are ready to communicate before data transfer begins."
]

# Embed all documents
doc_embeddings = [get_embedding(doc) for doc in documents]

def retrieve(query_embedding: np.ndarray, top_k: int = 3) -> list[tuple[str, float]]:
    """Retrieve top-k documents by cosine similarity."""
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    ranked = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Compare standard vs HyDE retrieval
print("=== Standard Retrieval (embed query directly) ===")
query_embedding = get_embedding(query)
standard_results = retrieve(query_embedding)
for doc, score in standard_results:
    print(f"[{score:.3f}] {doc[:80]}...")

print("\n=== HyDE Retrieval (embed hypothetical answer) ===")
hyde_embedding = get_embedding(hypothetical)
hyde_results = retrieve(hyde_embedding)
for doc, score in hyde_results:
    print(f"[{score:.3f}] {doc[:80]}...")

### When HyDE Helps

| Scenario | HyDE Benefit |
|----------|-------------|
| Question → Answer retrieval | High - bridges the query-document gap |
| Vague/short queries | High - LLM expands intent |
| Technical domains | Medium - generates domain vocabulary |
| Keyword search | Low - direct matching often better |
| Already well-formed queries | Low - overhead not worth it |

### Trade-offs

- **Latency**: Extra LLM call per query (~200-500ms)
- **Cost**: Additional tokens for generation
- **Hallucination risk**: Hypothetical might include wrong terms that retrieve irrelevant docs
- **Works best**: When queries are questions and documents are answers

In [ ]:
@dataclass
class HyDERetriever:
    """Retriever using Hypothetical Document Embeddings."""
    documents: list[str]
    doc_embeddings: list[np.ndarray] = field(default_factory=list)
    
    def __post_init__(self):
        """Embed documents on initialization."""
        if not self.doc_embeddings:
            print(f"Embedding {len(self.documents)} documents...")
            self.doc_embeddings = [get_embedding(doc) for doc in self.documents]
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 3, 
        use_hyde: bool = True
    ) -> list[dict]:
        """Retrieve documents, optionally using HyDE."""
        
        if use_hyde:
            # Generate hypothetical answer
            hypothetical = generate_hypothetical_answer(query)
            query_embedding = get_embedding(hypothetical)
            method = "hyde"
        else:
            query_embedding = get_embedding(query)
            hypothetical = None
            method = "standard"
        
        # Compute similarities
        similarities = [
            cosine_similarity(query_embedding, doc_emb) 
            for doc_emb in self.doc_embeddings
        ]
        
        # Rank and return
        ranked_indices = np.argsort(similarities)[::-1][:top_k]
        
        return {
            "method": method,
            "hypothetical": hypothetical,
            "results": [
                {
                    "document": self.documents[i],
                    "score": similarities[i],
                    "rank": rank + 1
                }
                for rank, i in enumerate(ranked_indices)
            ]
        }

# Test the retriever
retriever = HyDERetriever(documents=documents, doc_embeddings=doc_embeddings)

# Compare on a few queries
test_queries = [
    "What happens when packets get lost?",
    "How do computers establish a connection?",
    "Why is UDP faster?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    
    # Standard
    standard = retriever.retrieve(q, top_k=1, use_hyde=False)
    print(f"  Standard: [{standard['results'][0]['score']:.3f}] {standard['results'][0]['document'][:60]}...")
    
    # HyDE
    hyde = retriever.retrieve(q, top_k=1, use_hyde=True)
    print(f"  HyDE:     [{hyde['results'][0]['score']:.3f}] {hyde['results'][0]['document'][:60]}...")

---

## Part 2: Parent Document Retrieval

### The Chunking Dilemma

- **Small chunks** (100-200 tokens): Precise matching, but lose context
- **Large chunks** (1000+ tokens): Rich context, but noisy matching and diluted relevance

### The Solution: Index Small, Return Large

1. Split documents into a **hierarchy**: Document → Sections → Paragraphs → Sentences
2. **Index** the small chunks (paragraphs/sentences) for precise matching
3. **Return** the parent chunk (section/document) for rich context

This gives you precise retrieval *and* sufficient context for the LLM.

In [ ]:
# Sample document with clear hierarchy
SAMPLE_DOCUMENT = """
# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data. 
Rather than being explicitly programmed, these systems identify patterns and make decisions with minimal human intervention.
The field has grown rapidly due to increased computational power and data availability.

## Supervised Learning

Supervised learning uses labeled training data to learn a mapping from inputs to outputs.
Common algorithms include linear regression, decision trees, and neural networks.
The model is trained on examples where the correct answer is known, then tested on unseen data.

### Classification

Classification predicts discrete categories. Examples include spam detection (spam/not spam) 
and image recognition (cat/dog/bird). Evaluation metrics include accuracy, precision, recall, and F1 score.

### Regression

Regression predicts continuous values. Examples include house price prediction and stock forecasting.
Evaluation metrics include MSE (Mean Squared Error) and R-squared.

## Unsupervised Learning

Unsupervised learning finds patterns in unlabeled data. The algorithm discovers structure without guidance.
This is useful when you don't know what patterns exist in your data.

### Clustering

Clustering groups similar items together. K-means is a popular algorithm that partitions data into K clusters.
Applications include customer segmentation and document organization.

### Dimensionality Reduction

Dimensionality reduction simplifies data by reducing the number of features.
PCA (Principal Component Analysis) projects data onto principal components that capture the most variance.
This helps with visualization and removes redundant features.
"""

print(f"Document length: {len(SAMPLE_DOCUMENT)} characters")

In [ ]:
import re
from typing import Optional

@dataclass
class Chunk:
    """A chunk of text with hierarchy information."""
    id: str
    text: str
    level: str  # 'document', 'section', 'subsection', 'paragraph'
    parent_id: Optional[str] = None
    children_ids: list[str] = field(default_factory=list)
    embedding: Optional[np.ndarray] = None

def create_hierarchical_chunks(document: str) -> dict[str, Chunk]:
    """Parse a markdown document into hierarchical chunks."""
    chunks = {}
    
    # Create root document chunk
    doc_id = "doc_0"
    chunks[doc_id] = Chunk(
        id=doc_id,
        text=document.strip(),
        level="document"
    )
    
    # Split by ## (sections)
    sections = re.split(r'\n## ', document)
    current_parent = doc_id
    
    for i, section in enumerate(sections[1:], 1):  # Skip the intro
        section_id = f"section_{i}"
        section_text = "## " + section.strip()
        
        chunks[section_id] = Chunk(
            id=section_id,
            text=section_text,
            level="section",
            parent_id=doc_id
        )
        chunks[doc_id].children_ids.append(section_id)
        
        # Split section by ### (subsections)
        subsections = re.split(r'\n### ', section)
        
        for j, subsection in enumerate(subsections[1:], 1):  # Skip section header
            subsection_id = f"section_{i}_sub_{j}"
            subsection_text = "### " + subsection.strip()
            
            chunks[subsection_id] = Chunk(
                id=subsection_id,
                text=subsection_text,
                level="subsection",
                parent_id=section_id
            )
            chunks[section_id].children_ids.append(subsection_id)
            
            # Split subsection into paragraphs (sentences)
            paragraphs = [p.strip() for p in subsection_text.split('\n\n') if p.strip()]
            
            for k, para in enumerate(paragraphs):
                if para.startswith('###'):
                    continue  # Skip the header
                para_id = f"section_{i}_sub_{j}_para_{k}"
                
                chunks[para_id] = Chunk(
                    id=para_id,
                    text=para,
                    level="paragraph",
                    parent_id=subsection_id
                )
                chunks[subsection_id].children_ids.append(para_id)
    
    return chunks

# Create hierarchy
chunks = create_hierarchical_chunks(SAMPLE_DOCUMENT)

# Show hierarchy
print("Document Hierarchy:")
for chunk_id, chunk in chunks.items():
    indent = {"document": 0, "section": 1, "subsection": 2, "paragraph": 3}[chunk.level]
    preview = chunk.text[:50].replace('\n', ' ') + "..."
    print(f"{'  ' * indent}[{chunk.level}] {chunk_id}: {preview}")

In [ ]:
@dataclass
class ParentDocumentRetriever:
    """Index small chunks, return parent documents."""
    chunks: dict[str, Chunk]
    index_level: str = "paragraph"  # Level to index (small chunks)
    return_level: str = "subsection"  # Level to return (larger context)
    
    def __post_init__(self):
        """Embed chunks at the index level."""
        self.indexed_chunks = [
            chunk for chunk in self.chunks.values() 
            if chunk.level == self.index_level
        ]
        
        print(f"Embedding {len(self.indexed_chunks)} {self.index_level}-level chunks...")
        for chunk in self.indexed_chunks:
            chunk.embedding = get_embedding(chunk.text)
    
    def _get_parent_at_level(self, chunk_id: str, target_level: str) -> Chunk:
        """Traverse up the hierarchy to find parent at target level."""
        chunk = self.chunks[chunk_id]
        
        while chunk.level != target_level and chunk.parent_id:
            chunk = self.chunks[chunk.parent_id]
        
        return chunk
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 3,
        return_parent: bool = True
    ) -> list[dict]:
        """Retrieve by matching small chunks, returning larger context."""
        
        query_embedding = get_embedding(query)
        
        # Score all indexed chunks
        scored = []
        for chunk in self.indexed_chunks:
            score = cosine_similarity(query_embedding, chunk.embedding)
            scored.append((chunk, score))
        
        # Sort by score
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Get unique parents (avoid duplicates)
        seen_parents = set()
        results = []
        
        for chunk, score in scored:
            if return_parent:
                parent = self._get_parent_at_level(chunk.id, self.return_level)
                if parent.id in seen_parents:
                    continue
                seen_parents.add(parent.id)
                
                results.append({
                    "matched_chunk": chunk.text,
                    "matched_chunk_id": chunk.id,
                    "returned_content": parent.text,
                    "returned_chunk_id": parent.id,
                    "score": score
                })
            else:
                results.append({
                    "matched_chunk": chunk.text,
                    "matched_chunk_id": chunk.id,
                    "returned_content": chunk.text,
                    "returned_chunk_id": chunk.id,
                    "score": score
                })
            
            if len(results) >= top_k:
                break
        
        return results

# Create retriever
parent_retriever = ParentDocumentRetriever(
    chunks=chunks,
    index_level="paragraph",
    return_level="subsection"
)

In [ ]:
# Test parent document retrieval
query = "How do you measure classification performance?"

print(f"Query: {query}\n")

# Without parent retrieval (just the matched paragraph)
print("=== Standard Retrieval (return matched chunk only) ===")
results = parent_retriever.retrieve(query, top_k=1, return_parent=False)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(f"Matched: {r['matched_chunk']}")
    print(f"Returned: {r['returned_content']}")

print("\n=== Parent Document Retrieval (return subsection) ===")
results = parent_retriever.retrieve(query, top_k=1, return_parent=True)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(f"Matched paragraph: {r['matched_chunk'][:80]}...")
    print(f"\nReturned subsection:\n{r['returned_content']}")

### When to Use Parent Document Retrieval

| Scenario | Standard Chunks | Parent Doc Retrieval |
|----------|----------------|---------------------|
| Q&A with short answers | ✓ | |
| Complex questions needing context | | ✓ |
| Legal/medical docs (precision matters) | | ✓ |
| Simple keyword lookup | ✓ | |
| Multi-hop reasoning | | ✓ |

### Trade-offs

- **More context**: LLM gets richer information to work with
- **Potential noise**: Parent may include irrelevant information
- **Deduplication needed**: Multiple child chunks may match the same parent
- **Storage overhead**: Must maintain chunk hierarchy

---

## Part 3: Combining HyDE + Parent Document Retrieval

These techniques are complementary:
- **HyDE** improves the *query side* (better query representation)
- **Parent Doc** improves the *document side* (better context delivery)

Let's combine them.

In [ ]:
@dataclass
class AdvancedRetriever:
    """Combined HyDE + Parent Document retrieval."""
    chunks: dict[str, Chunk]
    index_level: str = "paragraph"
    return_level: str = "subsection"
    indexed_chunks: list[Chunk] = field(default_factory=list)
    
    def __post_init__(self):
        """Embed chunks at the index level."""
        self.indexed_chunks = [
            chunk for chunk in self.chunks.values() 
            if chunk.level == self.index_level
        ]
        
        # Only embed if not already done
        needs_embedding = [c for c in self.indexed_chunks if c.embedding is None]
        if needs_embedding:
            print(f"Embedding {len(needs_embedding)} chunks...")
            for chunk in needs_embedding:
                chunk.embedding = get_embedding(chunk.text)
    
    def _get_parent_at_level(self, chunk_id: str, target_level: str) -> Chunk:
        chunk = self.chunks[chunk_id]
        while chunk.level != target_level and chunk.parent_id:
            chunk = self.chunks[chunk.parent_id]
        return chunk
    
    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        use_hyde: bool = False,
        return_parent: bool = True
    ) -> dict:
        """Retrieve with optional HyDE and parent document return."""
        
        # Step 1: Get query embedding (with or without HyDE)
        hypothetical = None
        if use_hyde:
            hypothetical = generate_hypothetical_answer(query)
            query_embedding = get_embedding(hypothetical)
        else:
            query_embedding = get_embedding(query)
        
        # Step 2: Score chunks
        scored = []
        for chunk in self.indexed_chunks:
            score = cosine_similarity(query_embedding, chunk.embedding)
            scored.append((chunk, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Step 3: Collect results (with optional parent expansion)
        seen_parents = set()
        results = []
        
        for chunk, score in scored:
            if return_parent:
                parent = self._get_parent_at_level(chunk.id, self.return_level)
                if parent.id in seen_parents:
                    continue
                seen_parents.add(parent.id)
                returned_content = parent.text
                returned_id = parent.id
            else:
                returned_content = chunk.text
                returned_id = chunk.id
            
            results.append({
                "matched_chunk": chunk.text,
                "returned_content": returned_content,
                "returned_id": returned_id,
                "score": score
            })
            
            if len(results) >= top_k:
                break
        
        return {
            "query": query,
            "hypothetical": hypothetical,
            "use_hyde": use_hyde,
            "return_parent": return_parent,
            "results": results
        }

# Create combined retriever (reuses existing embeddings)
advanced_retriever = AdvancedRetriever(
    chunks=chunks,
    index_level="paragraph",
    return_level="subsection"
)

In [ ]:
# Compare all four combinations
query = "What algorithm groups similar things together?"

print(f"Query: {query}\n")
print("=" * 60)

configs = [
    {"use_hyde": False, "return_parent": False, "name": "Standard (no HyDE, no parent)"},
    {"use_hyde": True, "return_parent": False, "name": "HyDE only"},
    {"use_hyde": False, "return_parent": True, "name": "Parent Doc only"},
    {"use_hyde": True, "return_parent": True, "name": "HyDE + Parent Doc"},
]

for config in configs:
    print(f"\n### {config['name']} ###")
    result = advanced_retriever.retrieve(
        query, 
        top_k=1, 
        use_hyde=config["use_hyde"],
        return_parent=config["return_parent"]
    )
    
    if result["hypothetical"]:
        print(f"Hypothetical: {result['hypothetical'][:100]}...")
    
    top = result["results"][0]
    print(f"Score: {top['score']:.3f}")
    print(f"Matched: {top['matched_chunk'][:80]}...")
    print(f"Returned ({len(top['returned_content'])} chars): {top['returned_content'][:100]}...")

---

## Part 4: RAG Integration

Let's integrate our advanced retriever into a complete RAG pipeline.

In [ ]:
def rag_with_advanced_retrieval(
    question: str,
    retriever: AdvancedRetriever,
    use_hyde: bool = True,
    return_parent: bool = True,
    top_k: int = 2
) -> dict:
    """Complete RAG pipeline with advanced retrieval."""
    
    # Step 1: Retrieve
    retrieval = retriever.retrieve(
        question,
        top_k=top_k,
        use_hyde=use_hyde,
        return_parent=return_parent
    )
    
    # Step 2: Format context
    context = "\n\n---\n\n".join([
        r["returned_content"] for r in retrieval["results"]
    ])
    
    # Step 3: Generate answer
    response = client.responses.create(
        model=MODEL,
        instructions="""Answer the question based on the provided context.
If the context doesn't contain enough information, say so.
Be concise and accurate.""",
        input=f"""Context:
{context}

Question: {question}

Answer:"""
    )
    
    return {
        "question": question,
        "answer": response.output_text,
        "retrieval": retrieval,
        "context_length": len(context)
    }

# Test RAG pipeline
questions = [
    "What's the difference between classification and regression?",
    "How do you reduce the number of features in a dataset?",
    "What is K-means used for?"
]

for q in questions:
    print(f"\nQ: {q}")
    result = rag_with_advanced_retrieval(q, advanced_retriever)
    print(f"A: {result['answer']}")
    print(f"   (Retrieved {len(result['retrieval']['results'])} chunks, {result['context_length']} chars)")

---

## Guidelines: When to Use What

| Technique | Use When | Avoid When |
|-----------|----------|------------|
| **HyDE** | Queries are questions; docs are answers | Keyword search; exact matching needed |
| **Parent Doc** | Context is important; docs have hierarchy | Simple lookups; storage is constrained |
| **Both** | Complex Q&A over structured documents | Latency-critical applications |
| **Neither** | Fast keyword search; well-formed queries | - |

### Cost/Latency Comparison

| Method | Extra LLM Calls | Extra Embeddings | Latency Impact |
|--------|----------------|------------------|----------------|
| Standard | 0 | 0 | Baseline |
| HyDE | +1 per query | +1 per query | +200-500ms |
| Parent Doc | 0 | 0 | Negligible |
| HyDE + Parent | +1 per query | +1 per query | +200-500ms |

---

## Exercises

### Exercise 1: Multi-HyDE

Generate multiple hypothetical documents and average their embeddings for better coverage.

In [ ]:
def multi_hyde_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    n_hypotheticals: int = 3,
    top_k: int = 3
) -> list[dict]:
    """Generate multiple hypothetical answers, average their embeddings."""
    # Your implementation here
    pass

### Exercise 2: Adaptive Parent Level

Choose the parent level dynamically based on query complexity. Simple queries get smaller parents, complex queries get larger parents.

In [ ]:
def adaptive_parent_retrieve(
    query: str,
    retriever: AdvancedRetriever,
    top_k: int = 3
) -> list[dict]:
    """Dynamically choose parent level based on query complexity."""
    # Your implementation here
    pass

### Exercise 3: HyDE with Domain Prompting

Customize the HyDE generation prompt based on the domain (technical docs, legal, medical, etc.).

In [ ]:
def domain_aware_hyde(
    query: str,
    domain: str,  # "technical", "legal", "medical", "general"
) -> str:
    """Generate hypothetical document tailored to the domain."""
    # Your implementation here
    pass

---

## Summary

- **HyDE** bridges the query-document gap by generating hypothetical answers before embedding
- **Parent Document Retrieval** indexes small chunks for precision but returns larger chunks for context
- **Combined** gives you both better matching AND richer context
- **Trade-off**: Extra latency and cost vs. improved retrieval quality

Use these techniques when retrieval quality matters more than latency, especially for complex Q&A over structured documents.

In the next notebook, we'll learn how to measure retrieval quality with metrics like Precision, Recall, NDCG, and MAP, plus implement MMR for diverse results.